# S1 — Foundations of LLMs: Ejercicios de Código

**Módulo:** Semestre 1 — Foundations of LLMs  
**Lecciones de referencia:** 1.2 (Tokens y Context Windows), 1.4 (Temperatura y Determinismo), 1.5 (Fine-tuning vs RAG), 1.7 (Coste Real en Producción)

## Objetivo

Estos ejercicios convierten los modelos mentales del S1 en código ejecutable. Cada ejercicio implementa una pieza de infraestructura real que usarías en un sistema de producción con LLMs.

## Instrucciones

1. Ejecuta la celda de setup primero.
2. Implementa cada función donde dice `# TODO` o `raise NotImplementedError`.
3. Ejecuta la celda de tests del ejercicio. Si pasan sin excepción, el ejercicio está bien.
4. Los tests usan mocks — **no necesitas API key real para que pasen**.
5. Si tienes API key, cambia `USE_MOCK = True` a `False` en el setup para probar contra la API real.

---
**Dificultad:** E1 (básico) → E4 (integra todo el módulo)

In [ ]:
# Setup — ejecuta esta celda primero
# pip install openai tiktoken numpy  (descomenta si no están instalados)
# !pip install openai tiktoken numpy

import os
import hashlib
import math
from typing import List, Dict, Tuple, Optional
from unittest.mock import MagicMock, patch

import tiktoken

# ── API config ──────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'sk-test')
USE_MOCK = True  # Cambia a False para usar la API real

# ── Precio de referencia ($/1K tokens) ──────────────────────────────────────
MODEL_PRICING = {
    'gpt-4o': {'input': 0.0025, 'output': 0.010},
    'gpt-4o-mini': {'input': 0.00015, 'output': 0.0006},
    'gpt-3.5-turbo': {'input': 0.0005, 'output': 0.0015},
}

print('Setup OK. USE_MOCK =', USE_MOCK)

---
## Ejercicio 1 — Medir variabilidad por temperatura

**Contexto (Lección 1.4):** La temperatura controla cuánta aleatoriedad introduces en el sampling de tokens. Temperatura 0 = siempre el token más probable = output predecible. Temperatura alta = distribución más plana = outputs más variados.

**Qué implementar:**
- `run_with_temperature(prompt, temperature, n_runs)` — ejecuta el mismo prompt N veces y devuelve la lista de outputs.
- `variability_score(outputs)` — calcula qué fracción de los outputs son únicos (0 = todos iguales, 1 = todos distintos).

**Criterios de éxito:**
- Con temperatura 0 → `variability_score` ≤ 0.2 (muy pocos outputs distintos).
- Con temperatura 1.0 → `variability_score` > 0.4 (variación notable).
- La función maneja correctamente listas de 1 elemento (score = 0.0).

In [ ]:
def variability_score(outputs: List[str]) -> float:
    """
    Calcula la fracción de outputs únicos sobre el total.
    
    Args:
        outputs: lista de strings devueltos por el LLM.
    Returns:
        float en [0, 1]. 0 = todos iguales, 1 = todos distintos.
    
    Ejemplos:
        variability_score(['a', 'a', 'a'])   → 0.0
        variability_score(['a', 'b', 'a'])   → 0.333...
        variability_score(['a', 'b', 'c'])   → 1.0
        variability_score(['a'])             → 0.0  (un solo elemento: sin variación)
    """
    # TODO: implementa la fórmula. Pista: len(set(outputs)) / len(outputs)
    # Maneja el edge case de lista vacía o un solo elemento.
    raise NotImplementedError


def run_with_temperature(
    prompt: str,
    temperature: float,
    n_runs: int = 5,
    model: str = 'gpt-4o-mini',
) -> List[str]:
    """
    Ejecuta el mismo prompt N veces con la temperatura dada.
    
    Args:
        prompt: texto del mensaje de usuario.
        temperature: valor entre 0 y 2.
        n_runs: número de ejecuciones.
        model: modelo a usar.
    Returns:
        Lista de strings con los outputs del LLM.
    """
    # TODO: implementa la llamada a la API n_runs veces.
    # Usa USE_MOCK si está activo (ver celda de tests).
    # Estructura de llamada real:
    #
    #   from openai import OpenAI
    #   client = OpenAI(api_key=OPENAI_API_KEY)
    #   response = client.chat.completions.create(
    #       model=model,
    #       messages=[{'role': 'user', 'content': prompt}],
    #       temperature=temperature,
    #   )
    #   return response.choices[0].message.content
    raise NotImplementedError

In [ ]:
# ── Tests Ejercicio 1 ────────────────────────────────────────────────────────

# Test variability_score (sin API)
assert variability_score(['a', 'a', 'a']) == 0.0,          "Todos iguales → 0.0"
assert variability_score(['a']) == 0.0,                     "Un solo elemento → 0.0"
assert variability_score(['a', 'b', 'c']) == 1.0,           "Todos distintos → 1.0"
assert abs(variability_score(['a', 'b', 'a']) - 2/3) < 0.01, "2 únicos de 3 → 0.666"

# Test run_with_temperature con mock
# Mock: temperatura 0 siempre devuelve la misma respuesta;
#       temperatura 1.0 devuelve respuestas distintas.
MOCK_RESPONSES_T0   = ['42'] * 5
MOCK_RESPONSES_T1   = ['42', 'cuarenta y dos', '42.', 'XLII', 'cuarentaidós']

def _mock_run(prompt, temperature, n_runs=5, model='gpt-4o-mini'):
    """Sustituto de run_with_temperature para tests sin API."""
    if temperature == 0:
        return MOCK_RESPONSES_T0[:n_runs]
    else:
        return MOCK_RESPONSES_T1[:n_runs]

if USE_MOCK:
    # Sustituye temporalmente para el test
    _original = run_with_temperature
    run_with_temperature_test = _mock_run
else:
    run_with_temperature_test = run_with_temperature

outputs_t0 = run_with_temperature_test('Di el número 42', temperature=0, n_runs=5)
outputs_t1 = run_with_temperature_test('Di el número 42', temperature=1.0, n_runs=5)

score_t0 = variability_score(outputs_t0)
score_t1 = variability_score(outputs_t1)

assert score_t0 <= 0.2,  f"Temperatura 0 → score esperado ≤ 0.2, obtenido {score_t0}"
assert score_t1 > 0.4,   f"Temperatura 1.0 → score esperado > 0.4, obtenido {score_t1}"

print(f"✅ Ejercicio 1 OK")
print(f"   Temperatura 0:   variability_score = {score_t0:.2f}  (outputs: {outputs_t0})")
print(f"   Temperatura 1.0: variability_score = {score_t1:.2f}  (outputs: {outputs_t1})")

---
## Ejercicio 2 — Calculadora de costes de LLM

**Contexto (Lección 1.7):** El coste de una llamada a LLM = (tokens entrada × precio entrada) + (tokens salida × precio salida). Los tokens de salida cuestan más. Modelar el coste antes de diseñar evita sorpresas en producción.

**Qué implementar:**
- `count_message_tokens(messages, model)` — cuenta los tokens de una lista de mensajes, incluyendo el overhead por mensaje.
- `calculate_cost(messages, model, output_tokens)` — devuelve el desglose completo de coste en USD.
- `estimate_monthly_cost(daily_conversations, avg_turns, avg_input_tokens, avg_output_tokens, model)` — proyección mensual.

**Criterios de éxito:**
- El conteo de tokens es preciso (tiktoken, sin API).
- `calculate_cost` devuelve un dict con las 5 claves esperadas.
- Los costes se calculan con la fórmula correcta: precio/1000 × tokens.

In [ ]:
def count_message_tokens(messages: List[Dict[str, str]], model: str = 'gpt-4o') -> int:
    """
    Cuenta los tokens de una lista de mensajes usando tiktoken.
    
    Overhead estándar de OpenAI:
        - 4 tokens por mensaje (role + separadores)
        - 2 tokens de 'priming' al final de la lista
    
    Args:
        messages: lista de dicts {'role': ..., 'content': ...}
        model: nombre del modelo (determina el encoding de tiktoken)
    Returns:
        int con el número total de tokens.
    """
    # TODO:
    # 1. Obtén el encoding con tiktoken.encoding_for_model(model)
    #    Si falla (modelo no reconocido), usa tiktoken.get_encoding('cl100k_base')
    # 2. Para cada mensaje, suma: len(encoding.encode(msg['content'])) + 4
    # 3. Añade 2 tokens de priming al total
    raise NotImplementedError


def calculate_cost(
    messages: List[Dict[str, str]],
    model: str = 'gpt-4o',
    output_tokens: int = 150,
) -> Dict[str, float]:
    """
    Calcula el coste de una llamada al LLM.
    
    Args:
        messages: mensajes de entrada (system + historial + user).
        model: modelo de MODEL_PRICING.
        output_tokens: tokens de salida reales o estimados (default 150).
    Returns:
        Dict con claves:
            'input_tokens'     → int
            'output_tokens'    → int
            'input_cost_usd'   → float
            'output_cost_usd'  → float
            'total_cost_usd'   → float
    """
    # TODO:
    # 1. Obtén el precio del modelo desde MODEL_PRICING
    #    Si el modelo no está, lanza ValueError con mensaje claro.
    # 2. Cuenta los tokens de entrada con count_message_tokens.
    # 3. Calcula: input_cost = (input_tokens / 1000) * precio_input
    #             output_cost = (output_tokens / 1000) * precio_output
    # 4. Devuelve el dict completo.
    raise NotImplementedError


def estimate_monthly_cost(
    daily_conversations: int,
    avg_turns: int = 3,
    avg_input_tokens: int = 1500,
    avg_output_tokens: int = 300,
    model: str = 'gpt-4o-mini',
) -> Dict[str, float]:
    """
    Proyecta el coste mensual de un sistema de conversación.
    
    Args:
        daily_conversations: número de conversaciones por día.
        avg_turns: llamadas al LLM por conversación.
        avg_input_tokens: tokens de entrada promedio por llamada.
        avg_output_tokens: tokens de salida promedio por llamada.
        model: modelo de MODEL_PRICING.
    Returns:
        Dict con 'monthly_calls', 'monthly_input_tokens',
        'monthly_output_tokens', 'monthly_cost_usd'.
    """
    # TODO:
    # monthly_calls = daily_conversations * avg_turns * 30
    # Multiplica por tokens promedio para el total mensual
    # Calcula coste con MODEL_PRICING
    raise NotImplementedError

In [ ]:
# ── Tests Ejercicio 2 ────────────────────────────────────────────────────────

# Test count_message_tokens — tiktoken local, sin API
msgs_simple = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user',   'content': 'What is 2+2?'},
]
token_count = count_message_tokens(msgs_simple, model='gpt-4o')
# "You are a helpful assistant." ≈ 6 tokens + 4 overhead = 10
# "What is 2+2?" ≈ 5 tokens + 4 overhead = 9
# + 2 priming = 21 tokens total (±2 por diferencias de encoding)
assert 18 <= token_count <= 25, f"Token count fuera de rango esperado: {token_count}"

# Test calculate_cost — fórmula correcta
cost = calculate_cost(msgs_simple, model='gpt-4o', output_tokens=100)
assert set(cost.keys()) == {'input_tokens', 'output_tokens', 'input_cost_usd', 'output_cost_usd', 'total_cost_usd'}, \
    f"Faltan claves: {cost.keys()}"
assert cost['output_tokens'] == 100
# gpt-4o output: 0.010/1K tokens → 100 tokens = 0.001 USD
assert abs(cost['output_cost_usd'] - 0.001) < 0.0001, f"Output cost incorrecto: {cost['output_cost_usd']}"
assert cost['total_cost_usd'] == cost['input_cost_usd'] + cost['output_cost_usd']

# Test ValueError para modelo desconocido
try:
    calculate_cost(msgs_simple, model='modelo-inventado')
    assert False, "Debería haber lanzado ValueError"
except ValueError:
    pass

# Test estimate_monthly_cost
monthly = estimate_monthly_cost(
    daily_conversations=100,
    avg_turns=3,
    avg_input_tokens=1500,
    avg_output_tokens=300,
    model='gpt-4o-mini',
)
assert monthly['monthly_calls'] == 100 * 3 * 30,            "monthly_calls incorrecto"
assert monthly['monthly_input_tokens'] == 9000 * 1500,      "monthly_input_tokens incorrecto"
assert monthly['monthly_output_tokens'] == 9000 * 300,      "monthly_output_tokens incorrecto"
# gpt-4o-mini: input=0.00015/1K, output=0.0006/1K
# input cost  = 13_500_000 / 1000 * 0.00015 = 2.025 USD
# output cost = 2_700_000  / 1000 * 0.0006  = 1.62  USD
# total = 3.645 USD
assert abs(monthly['monthly_cost_usd'] - 3.645) < 0.01, \
    f"monthly_cost_usd incorrecto: {monthly['monthly_cost_usd']}"

print(f"✅ Ejercicio 2 OK")
print(f"   Tokens del mensaje de prueba: {token_count}")
print(f"   Coste de la llamada: ${cost['total_cost_usd']:.6f}")
print(f"   Proyección mensual (100 conv/día, gpt-4o-mini): ${monthly['monthly_cost_usd']:.3f}/mes")

---
## Ejercicio 3 — Context Window Monitor

**Contexto (Lección 1.2):** La context window es la RAM del modelo. Cuando la conversación crece y no cabe, el sistema tiene que decidir qué cortar. La regla práctica: al llegar al 80% del límite, recortar el historial más antiguo preservando siempre el system prompt.

**Qué implementar:**
- `ContextWindowMonitor` — clase que mantiene el historial y controla el tamaño.
  - `add_message(role, content)` — añade un mensaje.
  - `total_tokens()` — tokens actuales del historial.
  - `check_overflow()` → `True` si supera el 80% del máximo.
  - `trim_history()` — elimina mensajes no-system hasta que `check_overflow()` sea `False`.
  - `get_messages()` → lista de mensajes actual.

**Criterios de éxito:**
- `trim_history()` nunca elimina el mensaje system.
- Tras `trim_history()`, `check_overflow()` devuelve `False`.
- Si no hay mensajes para eliminar (solo system), no lanza excepción.

In [ ]:
class ContextWindowMonitor:
    """
    Gestiona el historial de mensajes respetando el límite de la context window.
    
    El sistema prompt (role='system') NUNCA se elimina durante el trim.
    El recorte empieza por los mensajes más antiguos (índice más bajo, excluyendo system).
    
    Ejemplo de uso:
        monitor = ContextWindowMonitor(max_tokens=4000, model='gpt-4o')
        monitor.add_message('system', 'Eres un asistente...')
        monitor.add_message('user', 'Hola')
        monitor.add_message('assistant', 'Hola, ¿en qué puedo ayudarte?')
        if monitor.check_overflow():
            monitor.trim_history()
    """

    OVERFLOW_THRESHOLD = 0.8  # Alerta al 80% del límite

    def __init__(self, max_tokens: int, model: str = 'gpt-4o'):
        """
        Args:
            max_tokens: tamaño máximo de la context window.
            model: modelo para el encoding de tiktoken.
        """
        # TODO: inicializa self.max_tokens, self.model, y self._messages = []
        # Obtén el encoding de tiktoken (con fallback a cl100k_base si falla)
        raise NotImplementedError

    def add_message(self, role: str, content: str) -> None:
        """Añade un mensaje al historial."""
        # TODO: añade {'role': role, 'content': content} a self._messages
        raise NotImplementedError

    def total_tokens(self) -> int:
        """Tokens actuales del historial completo."""
        # TODO: usa count_message_tokens(self._messages, self.model)
        # Importa o referencia la función del Ejercicio 2
        raise NotImplementedError

    def check_overflow(self) -> bool:
        """
        Devuelve True si el historial supera el OVERFLOW_THRESHOLD del máximo.
        Un historial vacío nunca hace overflow.
        """
        # TODO: return self.total_tokens() > self.max_tokens * self.OVERFLOW_THRESHOLD
        raise NotImplementedError

    def trim_history(self) -> int:
        """
        Elimina mensajes no-system (de más antiguo a más reciente) hasta que
        check_overflow() sea False o no queden mensajes eliminables.
        
        Returns:
            Número de mensajes eliminados.
        """
        # TODO:
        # 1. Identifica el primer mensaje no-system por índice.
        # 2. Elimínalo de self._messages.
        # 3. Repite mientras check_overflow() sea True y haya mensajes eliminables.
        # 4. Devuelve cuántos mensajes se eliminaron.
        raise NotImplementedError

    def get_messages(self) -> List[Dict[str, str]]:
        """Devuelve copia del historial actual."""
        # TODO: devuelve una copia (no la referencia) de self._messages
        raise NotImplementedError

In [ ]:
# ── Tests Ejercicio 3 ────────────────────────────────────────────────────────

# Setup: monitor con límite pequeño para forzar overflow fácilmente
monitor = ContextWindowMonitor(max_tokens=200, model='gpt-4o')

# El system prompt ocupa pocos tokens — no debe ser eliminado nunca
monitor.add_message('system', 'Eres un asistente de soporte técnico.')
initial_tokens = monitor.total_tokens()
assert initial_tokens > 0, "El monitor debe contar tokens del system prompt"
assert not monitor.check_overflow(), "Solo el system prompt no debe hacer overflow"

# Añadir mensajes largos hasta forzar overflow
long_content = 'Esta es una respuesta larga de prueba con bastante contenido para consumir tokens. ' * 5
monitor.add_message('user', long_content)
monitor.add_message('assistant', long_content)
monitor.add_message('user', long_content)

tokens_before_trim = monitor.total_tokens()
assert monitor.check_overflow(), (
    f"Debería hacer overflow con {tokens_before_trim} tokens sobre un máximo de 200 "
    f"(threshold 80% = 160). Ajusta el contenido del test si tiktoken difiere."
)

# trim_history debe resolver el overflow
removed = monitor.trim_history()
assert removed > 0, "Debe haber eliminado al menos un mensaje"
assert not monitor.check_overflow(), (
    f"Tras trim_history(), check_overflow() debe ser False. "
    f"Tokens actuales: {monitor.total_tokens()}, máximo: 200, threshold: 160"
)

# El system prompt NUNCA se elimina
remaining = monitor.get_messages()
system_msgs = [m for m in remaining if m['role'] == 'system']
assert len(system_msgs) == 1, "El system prompt no debe eliminarse nunca"
assert system_msgs[0]['content'] == 'Eres un asistente de soporte técnico.'

# get_messages devuelve copia, no referencia
msgs_copy = monitor.get_messages()
msgs_copy.append({'role': 'user', 'content': 'extra'})
assert len(monitor.get_messages()) != len(msgs_copy), "get_messages debe devolver copia"

# Monitor vacío no hace overflow
empty_monitor = ContextWindowMonitor(max_tokens=100)
assert not empty_monitor.check_overflow(), "Monitor vacío nunca hace overflow"
empty_monitor.trim_history()  # No debe lanzar excepción

print(f"✅ Ejercicio 3 OK")
print(f"   Tokens antes del trim: {tokens_before_trim}")
print(f"   Mensajes eliminados: {removed}")
print(f"   Tokens después del trim: {monitor.total_tokens()}")
print(f"   Mensajes restantes: {len(monitor.get_messages())} (incluyendo system)")

---
## Ejercicio 4 — RAG mínimo con evaluación

**Contexto (Lección 1.5):** RAG = el conocimiento vive fuera del modelo. En lugar de meter todos los documentos en el contexto, recuperas solo los fragmentos relevantes para cada query. El modelo es el motor de síntesis; el vector store es la memoria.

**Qué implementar:**
- `embed(text)` — representación vectorial del texto (hash-based para tests sin API; reemplazable por embeddings reales).
- `cosine_similarity(a, b)` — similitud entre dos vectores.
- `retrieve(query, knowledge_base, top_k)` — devuelve los K documentos más similares a la query.
- `build_rag_prompt(question, retrieved_docs)` — construye el prompt con los documentos como contexto.
- `answer_with_rag(question, knowledge_base)` — pipeline completo: retrieve → prompt → LLM.

**Criterios de éxito:**
- `retrieve` devuelve exactamente `top_k` documentos cuando hay suficientes en la base.
- El documento más similar a la query aparece en posición 0.
- `build_rag_prompt` incluye los documentos recuperados en el texto del prompt.

In [ ]:
import math
import hashlib


def embed(text: str, dim: int = 64) -> List[float]:
    """
    Genera un vector de embedding para el texto dado.
    
    Implementación 'fake' para tests sin API: usa SHA-256 del texto
    para generar un vector determinista de dimensión `dim`.
    Los documentos con palabras similares tendrán vectores distintos
    (no es semántico), pero permite testear el pipeline sin OpenAI.
    
    Para producción real, reemplaza con:
        client.embeddings.create(input=text, model='text-embedding-3-small')
        return response.data[0].embedding
    
    Args:
        text: texto a embeddear.
        dim: dimensiones del vector.
    Returns:
        Lista de floats de longitud `dim`.
    """
    # TODO:
    # 1. Calcula el hash SHA-256 del texto en bytes.
    # 2. Usa los bytes del hash para generar `dim` floats en [-1, 1].
    #    Pista: repite el hash si dim > 32, convierte cada byte a float.
    #           byte_val / 255.0 * 2 - 1  → da un float en [-1, 1]
    # 3. Normaliza el vector (divide por su norma L2) para que la
    #    cosine similarity funcione correctamente.
    raise NotImplementedError


def cosine_similarity(a: List[float], b: List[float]) -> float:
    """
    Calcula la similitud coseno entre dos vectores.
    
    cosine_similarity(a, b) = dot(a, b) / (norm(a) * norm(b))
    
    Returns:
        float en [-1, 1]. 1 = idénticos, 0 = ortogonales, -1 = opuestos.
    """
    # TODO:
    # 1. Calcula el producto escalar (dot product) de a y b.
    # 2. Calcula las normas L2 de a y b.
    # 3. Devuelve dot / (norm_a * norm_b). Maneja norm = 0 devolviendo 0.0.
    raise NotImplementedError


def retrieve(
    query: str,
    knowledge_base: List[str],
    top_k: int = 3,
) -> List[Tuple[str, float]]:
    """
    Recupera los K documentos más similares a la query.
    
    Args:
        query: pregunta del usuario.
        knowledge_base: lista de strings (documentos).
        top_k: número de documentos a devolver.
    Returns:
        Lista de tuplas (documento, score) ordenada de mayor a menor score.
        Longitud: min(top_k, len(knowledge_base)).
    """
    # TODO:
    # 1. Embeddea la query con embed(query).
    # 2. Para cada documento, calcula embed(doc) y cosine_similarity(query_vec, doc_vec).
    # 3. Ordena los pares (doc, score) de mayor a menor score.
    # 4. Devuelve los primeros top_k.
    raise NotImplementedError


def build_rag_prompt(question: str, retrieved_docs: List[Tuple[str, float]]) -> str:
    """
    Construye el prompt para el LLM incluyendo los documentos recuperados.
    
    Formato esperado:
        Responde la siguiente pregunta basándote ÚNICAMENTE en los documentos
        proporcionados. Si la respuesta no está en los documentos, di que
        no tienes esa información.
        
        DOCUMENTOS:
        [1] <texto del doc 1>
        [2] <texto del doc 2>
        ...
        
        PREGUNTA: <question>
    
    Args:
        question: la pregunta original del usuario.
        retrieved_docs: lista de (doc_text, score) de retrieve().
    Returns:
        String con el prompt completo.
    """
    # TODO: construye el prompt con el formato indicado.
    raise NotImplementedError


def answer_with_rag(
    question: str,
    knowledge_base: List[str],
    top_k: int = 3,
    model: str = 'gpt-4o-mini',
) -> Dict:
    """
    Pipeline RAG completo: retrieve → build_prompt → LLM → answer.
    
    Args:
        question: pregunta del usuario.
        knowledge_base: lista de documentos.
        top_k: documentos a recuperar.
        model: modelo LLM.
    Returns:
        Dict con:
            'answer'     → str con la respuesta del LLM (o mock)
            'sources'    → list de (doc, score) recuperados
            'prompt'     → el prompt construido
    """
    # TODO:
    # 1. Llama a retrieve(question, knowledge_base, top_k).
    # 2. Construye el prompt con build_rag_prompt(question, retrieved).
    # 3. Si USE_MOCK: devuelve 'answer': 'MOCK_ANSWER'.
    #    Si no: llama al LLM con el prompt y devuelve la respuesta real.
    # 4. Devuelve el dict con 'answer', 'sources', 'prompt'.
    raise NotImplementedError

In [ ]:
# ── Tests Ejercicio 4 ────────────────────────────────────────────────────────

# Test embed: determinista y normalizado
v1 = embed('hola mundo')
v2 = embed('hola mundo')
v3 = embed('texto completamente diferente')

assert len(v1) == 64, f"El vector debe tener 64 dimensiones, tiene {len(v1)}"
assert v1 == v2, "embed() debe ser determinista: mismo texto → mismo vector"
assert v1 != v3, "Textos distintos deben producir vectores distintos"

# Vector normalizado: norma L2 ≈ 1.0
norm = math.sqrt(sum(x**2 for x in v1))
assert abs(norm - 1.0) < 0.001, f"El vector debe estar normalizado (norma≈1), norma={norm:.4f}"

# Test cosine_similarity
assert abs(cosine_similarity(v1, v1) - 1.0) < 0.001, "Similitud de un vector consigo mismo ≈ 1.0"
assert -1.0 <= cosine_similarity(v1, v3) <= 1.0, "Similitud debe estar en [-1, 1]"
assert cosine_similarity([0.0, 0.0], [1.0, 0.0]) == 0.0, "Vector cero → similitud 0.0"

# Test retrieve: base de conocimiento con documentos temáticos
# El embedding es hash-based, no semántico, así que usamos textos IDÉNTICOS para
# verificar que el documento exacto se recupera con score 1.0
kb = [
    'El precio del plan básico es 9.99 euros al mes.',
    'Para cancelar tu suscripción ve a Configuración > Cuenta > Cancelar.',
    'El soporte técnico está disponible de lunes a viernes de 9:00 a 18:00.',
    'El plan premium incluye almacenamiento ilimitado y soporte prioritario.',
    'La facturación se realiza el primer día de cada mes.',
]

# Recuperar el primer documento con su texto exacto → debe aparecer en posición 0 con score 1.0
query_exact = 'El precio del plan básico es 9.99 euros al mes.'
results = retrieve(query_exact, kb, top_k=3)

assert len(results) == 3, f"retrieve debe devolver top_k=3 resultados, devolvió {len(results)}"
assert results[0][0] == query_exact, "El documento idéntico a la query debe ser el primero"
assert abs(results[0][1] - 1.0) < 0.001, f"Score del documento idéntico debe ser ≈1.0, es {results[0][1]}"
# Los resultados deben estar ordenados de mayor a menor score
scores = [s for _, s in results]
assert scores == sorted(scores, reverse=True), "Los resultados deben estar ordenados por score desc"

# retrieve con top_k mayor que la base devuelve todos
all_results = retrieve(query_exact, kb, top_k=10)
assert len(all_results) == len(kb), "Si top_k > len(kb), devuelve todos los documentos"

# Test build_rag_prompt: el prompt debe incluir la pregunta y los documentos
prompt = build_rag_prompt('¿Cuánto cuesta el plan básico?', results)
assert '¿Cuánto cuesta el plan básico?' in prompt, "El prompt debe incluir la pregunta"
assert query_exact in prompt, "El prompt debe incluir el documento recuperado"
assert '[1]' in prompt, "El prompt debe numerar los documentos"

# Test answer_with_rag: pipeline completo (con mock)
result = answer_with_rag('¿Cuánto cuesta el plan básico?', kb, top_k=2)
assert set(result.keys()) >= {'answer', 'sources', 'prompt'}, f"Faltan claves: {result.keys()}"
assert len(result['sources']) == 2, "Debe devolver top_k=2 fuentes"
assert result['prompt'] == build_rag_prompt('¿Cuánto cuesta el plan básico?', result['sources'])
if USE_MOCK:
    assert result['answer'] == 'MOCK_ANSWER', "En modo mock, answer debe ser 'MOCK_ANSWER'"

print(f"✅ Ejercicio 4 OK")
print(f"   Dimensiones del embedding: {len(v1)}")
print(f"   Top-3 para '{query_exact[:40]}...':")
for doc, score in results:
    print(f"     [{score:.3f}] {doc[:60]}")

---
## Tips y referencias

### Para ir más lejos

**Ejercicio 1 — Temperatura:**
- Prueba con la API real y un prompt de extracción estructurada: compara variability_score con temperatura 0 vs 0.7.
- Experimenta con `top_p` (nucleus sampling): ¿produce variabilidad distinta a temperatura equivalente?
- Referencia: [OpenAI API - temperature](https://platform.openai.com/docs/api-reference/chat)

**Ejercicio 2 — Coste:**
- Añade soporte para `prompt_caching_discount` (80-90% de descuento en tokens repetidos).
- Integra la función con los precios actuales desde la API de OpenAI (cambian frecuentemente).
- Referencia: [OpenAI Pricing](https://openai.com/pricing)

**Ejercicio 3 — Context Window:**
- Implementa `summarize_and_compress()`: en lugar de eliminar mensajes, genera un resumen con el LLM.
- Añade una estrategia de prioridad: mensajes con `priority=high` no se eliminan antes que los normales.
- Referencia: Lección 5.5 del S5 (Context Compression).

**Ejercicio 4 — RAG:**
- Reemplaza `embed()` con `text-embedding-3-small` de OpenAI para búsqueda semántica real.
- Añade un reranker: tras el retrieval, usa un segundo LLM para puntuar la relevancia real.
- Mide `context_precision`: ¿qué % de los documentos recuperados era realmente útil?
- Referencia: Lección 5.3 y 5.4 del S5 (RAG y Chunking/Reranking).

### Reglas del S1 para llevar

| Situación | Decisión |
|---|---|
| Extracción estructurada, clasificación, validación | Temperatura 0 |
| Respuesta conversacional, resúmenes | Temperatura 0.3–0.7 |
| El conocimiento cambia o es grande | RAG, no fine-tuning |
| El comportamiento/estilo necesita ajuste | Fine-tuning |
| Error en producción, difícil de reproducir | Temperatura 0 + logging del prompt completo |
| Historial largo que no cabe | Trim por antigüedad + summarization |
| Elegir modelo | El más pequeño que resuelve el problema |
